# Samuel Custom Voice Fine-Tune — Kaggle (GPU T4 x2, never TPU)

> **Accelerator: GPU T4 x2** — SEANet `weight_norm` + causal `Conv1d` → XLA fails or recompiles every batch.

> **Pipeline:** Clone private repo → `prepare_custom_dataset.py` (manifest + pitch cache) → `torchrun DDP` → extract `last.pt` → re-export ONNX locally.

> **Source:** `kaggle/kaggle_train.py` is canonical; this notebook is generated.

**Kaggle Settings:** `T4 x2`, Internet ON, Persistence ON. Attach dataset `my-voice-wavs` (folder of `.wav`). Add Secret `GH_TOKEN` if repo private, `HF_TOKEN` to push.



In [ ]:
# Cell 1: Environment & Repo Setup — uv + private clone
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os, pathlib
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
!uv --version

GH_TOKEN = os.environ.get("GH_TOKEN","")
REPO = "lydorianP/samuel-realtime-parrot"
if GH_TOKEN:
    !git clone https://{GH_TOKEN}@github.com/{REPO}.git
else:
    !git clone https://github.com/{REPO}.git
%cd samuel-realtime-parrot
!uv python install 3.12
!uv sync

# Install vendor/samuel training deps (hydra, omegaconf, wandb, transformers, etc.)
!uv pip install -e vendor/samuel 2>&1 | tail -n 20
!uv run python -c "import hydra, omegaconf, wandb, transformers; print('deps ok')"
!ls -la


In [ ]:
# Cell 2: Dataset & Pitch Cache — uses repo's correct prepare script
import pathlib
WAV_DIR = pathlib.Path("/kaggle/input/my-voice-wavs")
if not WAV_DIR.exists():
    for p in pathlib.Path("/kaggle/input").glob("*"):
        if list(p.glob("*.wav")) or list(p.rglob("*.wav")):
            WAV_DIR = p
            print(f"Found {WAV_DIR}")
            break
print(f"WAV_DIR={WAV_DIR} wavs={len(list(WAV_DIR.rglob('*.wav'))) if WAV_DIR.exists() else 0}")

!uv run python scripts/prepare_custom_dataset.py \
    --wav-dir /kaggle/input/my-voice-wavs \
    --manifest manifests/custom.jsonl \
    --pitch-cache manifests/pitch_cache/custom_spf512.npz \
    --sample-rate 44100 --samples-per-frame 512

import numpy as np, json
print(open("manifests/custom.jsonl").readline()[:300])
d=np.load("manifests/pitch_cache/custom_spf512.npz")
print(list(d.files)[:8], "n_files", d["n_files"], "sr", d["sample_rate"], "spf", d["samples_per_frame"])


In [ ]:
# Cell 3: DDP Training on 2x T4 — batch 16 (64 OOMs), SSL 1.0 kept
!nvidia-smi
!torchrun --standalone --nproc_per_node=2 -m samuel.train \
    run.name=kaggle_custom_voice_ft \
    data.manifest_path=manifests/custom.jsonl \
    data.pitch_cache_path=manifests/pitch_cache/custom_spf512.npz \
    batch_size=16 optim.max_steps=5000 optim.warmup_steps=500 \
    log.eval_every=500 log.ckpt_every=1000 log.wandb_mode=offline
# If OOM: retry batch_size=8


In [ ]:
# Cell 4: Extract checkpoint for download
import shutil, pathlib, subprocess
ckpt_base = pathlib.Path("runs")
cands = sorted(ckpt_base.glob("kaggle_custom_voice_ft_*"))
ckpt_dir = (cands[-1]/"checkpoints") if cands else pathlib.Path("runs/kaggle_custom_voice_ft/checkpoints")
print(f"ckpt_dir={ckpt_dir} exists={ckpt_dir.exists()}")
for p in sorted(ckpt_dir.glob("*.pt")):
    print(p, f"{p.stat().st_size/1024/1024:.1f} MB")
if ckpt_dir.exists():
    latest = max(ckpt_dir.glob("*.pt"), key=lambda p: p.stat().st_mtime)
    shutil.copy(latest, "/kaggle/working/samuel_custom_last.pt")
    src_cfg = latest.parent.parent/"config.json"
    if src_cfg.exists():
        shutil.copy(src_cfg, "/kaggle/working/custom_config.json")
    print(f"✅ /kaggle/working/samuel_custom_last.pt { pathlib.Path('/kaggle/working/samuel_custom_last.pt').stat().st_size/1024/1024:.1f} MB")
    subprocess.run(["cp", str(latest), "samuel_custom_last.pt"])
else:
    print("No checkpoint"); import os; os.system("ls -R runs | head -n 100")


In [ ]:
# Optional Cell 5: Push to HF for Windows WebGPU download
import os
HF_TOKEN=os.environ.get("HF_TOKEN")
HF_REPO="barbarabhb/samuel-realtime-parrot-custom"
if HF_TOKEN:
    from huggingface_hub import create_repo, upload_file
    import subprocess
    try:
        create_repo(HF_REPO, repo_type="model", private=True, exist_ok=True, token=HF_TOKEN)
        print(f"Ensured HF repo exists: https://huggingface.co/{HF_REPO}")
    except Exception as e:
        print(f"create_repo warning: {e}")
    try:
        subprocess.run(["huggingface-cli", "login", "--token", HF_TOKEN, "--add-to-git-credential"], check=False)
    except Exception:
        pass
    import subprocess as sp
    try:
        sp.run(["uv", "run", "hf", "upload", HF_REPO, "/kaggle/working/samuel_custom_last.pt", "--repo-type", "model"], check=True)
        sp.run(["uv", "run", "hf", "upload", HF_REPO, "/kaggle/working/custom_config.json", "--repo-type", "model"], check=True)
        print(f"Uploaded via hf CLI to hf:{HF_REPO}")
    except Exception as e:
        print(f"hf CLI upload failed ({e}), trying upload_file")
        try:
            upload_file(path_or_fileobj="/kaggle/working/samuel_custom_last.pt", path_in_repo="samuel_custom_last.pt", repo_id=HF_REPO, repo_type="model", token=HF_TOKEN)
            upload_file(path_or_fileobj="/kaggle/working/custom_config.json", path_in_repo="config.json", repo_id=HF_REPO, repo_type="model", token=HF_TOKEN)
            print(f"Uploaded via upload_file to https://huggingface.co/{HF_REPO}")
        except Exception as e2:
            print(f"HF push failed: {e2}")
    print(f"HF repo: https://huggingface.co/{HF_REPO}")
else:
    print("Set HF_TOKEN secret to push — notebook reads from os.environ.get('HF_TOKEN'), never hardcoded")
    print("Add HF_TOKEN in Kaggle UI: Notebook → Add-ons → Secrets → Add HF_TOKEN")
